In [ ]:
from openai import OpenAI  # 或 import openai

In [ ]:
# 定义评估评分系统常量
SCORE_FULL = 1.0     # 完全匹配或完全令人满意
SCORE_PARTIAL = 0.5  # 部分匹配或部分令人满意
SCORE_NONE = 0.0     # 无匹配或不令人满意

In [ ]:
# 定义严格的评估提示模板
FAITHFULNESS_PROMPT_TEMPLATE = """
评估AI回答与真实答案的忠实度。

用户查询: {question}
AI回答: {response}
真实答案: {true_answer}

忠实度衡量AI回答与真实答案中事实的吻合程度，是否存在幻觉。

评分标准:
- 严格使用以下分值:
  * {full} = 完全忠实，与真实答案无矛盾
  * {partial} = 部分忠实，存在轻微矛盾
  * {none} = 不忠实，存在重大矛盾或幻觉

- 仅返回数值评分({full}, {partial}, 或 {none})，无需解释或附加文本。
"""


In [ ]:
RELEVANCY_PROMPT_TEMPLATE = """
评估AI回答与用户查询的相关性。

用户查询: {question}
AI回答: {response}

相关性衡量回答对用户问题的针对程度。

评分标准:
- 严格使用以下分值:
  * {full} = 完全相关，直接回答了问题
  * {partial} = 部分相关，涉及部分内容
  * {none} = 不相关，未回答问题

- 仅返回数值评分({full}, {partial}, 或 {none})，无需解释或附加文本。
"""


In [ ]:
def evaluate_response(question, response, true_answer):
    """
    根据忠实性和相关性评估AI生成的回答质量。

    参数:
    question (str): 用户的原始问题。
    response (str): 正在评估的AI生成的回答。
    true_answer (str): 用作基准的真实答案。

    返回:
    Tuple[float, float]: 包含(faithfulness_score, relevancy_score)的元组。
                         每个分数为: 1.0 (完全)，0.5 (部分)，或 0.0 (无)。
    """
    # 格式化评估提示
    faithfulness_prompt = FAITHFULNESS_PROMPT_TEMPLATE.format(
            question=question, 
            response=response, 
            true_answer=true_answer,
            full=SCORE_FULL,
            partial=SCORE_PARTIAL,
            none=SCORE_NONE
    )
    
    relevancy_prompt = RELEVANCY_PROMPT_TEMPLATE.format(
            question=question, 
            response=response,
            full=SCORE_FULL,
            partial=SCORE_PARTIAL,
            none=SCORE_NONE
    )

    # 请求模型进行忠实性评估
    faithfulness_response = client.chat.completions.create(
           model="gpt-3.5-turbo",
            temperature=0,
            messages=[
                    {"role": "system", "content": "You are an objective evaluator. Return ONLY the numerical score."},
                    {"role": "user", "content": faithfulness_prompt}
            ]
    )
    
    # 请求模型进行相关性评估
    relevancy_response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            temperature=0,
            messages=[
                    {"role": "system", "content": "You are an objective evaluator. Return ONLY the numerical score."},
                    {"role": "user", "content": relevancy_prompt}
            ]
    )
    
    # 提取分数并处理潜在的解析错误
    try:
            faithfulness_score = float(faithfulness_response.choices[0].message.content.strip())
    except ValueError:
            print("Warning: Could not parse faithfulness score, defaulting to 0")
            faithfulness_score = 0.0
            
    try:
            relevancy_score = float(relevancy_response.choices[0].message.content.strip())
    except ValueError:
            print("Warning: Could not parse relevancy score, defaulting to 0")
            relevancy_score = 0.0

    return faithfulness_score, relevancy_score

# 第一个验证数据的真实答案
true_answer = data[3]['ideal_answer']

# 对于块大小 256 和 128 评估回复
faithfulness, relevancy = evaluate_response(query, ai_responses_dict[256], true_answer)
faithfulness2, relevancy2 = evaluate_response(query, ai_responses_dict[128], true_answer)

# 打印评估分数
print(f"Faithfulness Score (Chunk Size 256): {faithfulness}")
print(f"Relevancy Score (Chunk Size 256): {relevancy}")

print(f"\n")

print(f"Faithfulness Score (Chunk Size 128): {faithfulness2}")
print(f"Relevancy Score (Chunk Size 128): {relevancy2}")